In [2]:
import cv2
import easyocr
import gradio as gr
import os

from PIL import Image

In [3]:
reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.
C:\Users\Localws\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


In [4]:
def preprocess_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    gray = cv2.GaussianBlur(gray, (3,3), 0)

    thresh = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )[1]
    return thresh

In [5]:
def extract_text(image):
    processed = preprocess_image(image)

    results = reader.readtext(processed)

    text = ""

    for result in results:
        text += result[1] + "\n"
    return text

In [6]:
def save_text(text):

    os.makedirs("output", exist_ok=True)

    filename = "output/extracted_text.txt"

    with open(filename,"w",encoding="utf-8") as file:
        file.write(text)

    return filename

In [7]:
def ocr_document(image):

    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    text = extract_text(image)

    file = save_text(text)

    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB), text, file

In [8]:
demo = gr.Interface(

    fn=ocr_document,

    inputs=gr.Image(type="numpy"),

    outputs=[
        gr.Image(label="Original Image"),
        gr.Textbox(label="Extracted Text", lines=15),
        gr.File(label="Download Text")
    ],

    title=" Document OCR Application",

    description="Upload a document image and extract readable text using EasyOCR."
)

In [9]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
!pip install pyngrok



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from pyngrok import ngrok

ngrok.set_auth_token("3H5BM898vO17BIQvMYDFYFdKrDe_4TrTW4Njpaz1ZYT2xoirX")

In [13]:
from pyngrok import ngrok

public_url = ngrok.connect(7860)

print(public_url.public_url)

https://confiding-distaste-facecloth.ngrok-free.dev
